# Phase 3 - Feature Signal EDA

## tl;dr

This notebook screens candidate feature families suggested by Phase 2: boundary/endpoint extrapolation, recent known-zone slopes, GR availability, typewell alignment, and trajectory position. It is diagnostic EDA, not a production model selection notebook.

## Context & Methods

The competition target zone is a long trailing block where `TVT_input` is missing. Phase 3 therefore focuses on features that are available at inference time and can explain target-zone `TVT` or target-zone delta from the last known TVT.

### Key Assumptions

- Visible local test wells are submission-shape examples only because Phase 2 found they overlap train wells.
- Train-only formation columns remain excluded from feature candidates.
- Linear probes below train and validate on sampled target-zone rows from different wells. They measure diagnostic signal, not final leaderboard-ready training protocol.


In [1]:
"Configure imports, paths, and deterministic parameters."

from pathlib import Path
import json
import math
import sys
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d
from scipy.stats import ks_2samp
from sklearn.feature_selection import mutual_info_regression
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = next(parent for parent in Path.cwd().parents if (parent / "src").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.io.loaders import discover_wells, load_horizontal_well, load_typewell
from src.evaluation.validation import group_well_folds
from src.utils.run_bundle import RunBundle

warnings.filterwarnings("ignore", category=RuntimeWarning)

RUN_TAG = "eda_phase3_feature_signal"
RUN_DATE = "2026-06-21"
SEED = 42
MAX_TARGET_ROWS_PER_WELL = 250
MAX_EXPORT_ROWS_PER_WELL = 50
MAX_MI_ROWS = 80000
N_SPLITS = 3

DATA_DIR = ROOT / "data" / "raw"
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"
ARTIFACT_DIR = ROOT / "artifacts"
TABLE_DIR = ARTIFACT_DIR / "tables"
FIGURE_DIR = ARTIFACT_DIR / "figures"
REPORT_DIR = ARTIFACT_DIR / "reports"

for directory in [TABLE_DIR, FIGURE_DIR, REPORT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)
plt.style.use("seaborn-v0_8-whitegrid")


In [2]:
"Define feature-screening helpers."


def safe_numeric(series):
    return pd.to_numeric(series, errors="coerce").astype(float)


def clean_gr(series):
    gr = safe_numeric(series).clip(lower=0, upper=300)
    return gr.fillna(gr.rolling(21, center=True, min_periods=1).median()).ffill().bfill()


def window_slope(md, values, end_pos, window):
    start_pos = max(0, end_pos - window + 1)
    x0 = float(md.iloc[start_pos])
    x1 = float(md.iloc[end_pos])
    y0 = float(values.iloc[start_pos])
    y1 = float(values.iloc[end_pos])
    if not np.isfinite([x0, x1, y0, y1]).all() or abs(x1 - x0) < 1e-9:
        return np.nan
    return (y1 - y0) / (x1 - x0)


def deterministic_sample_positions(mask, max_rows):
    positions = np.flatnonzero(mask.to_numpy())
    if len(positions) <= max_rows:
        return positions
    return positions[np.linspace(0, len(positions) - 1, max_rows).astype(int)]


def typewell_interpolator(typewell):
    if typewell.empty or "TVT" not in typewell.columns or "GR" not in typewell.columns:
        return None
    tw = typewell[["TVT", "GR"]].dropna().sort_values("TVT")
    if len(tw) < 2:
        return None
    return interp1d(
        tw["TVT"].to_numpy(float),
        tw["GR"].to_numpy(float),
        bounds_error=False,
        fill_value=(float(tw["GR"].iloc[0]), float(tw["GR"].iloc[-1])),
    )


def known_typewell_corr(horizontal, typewell):
    interp = typewell_interpolator(typewell)
    if interp is None:
        return np.nan
    known = horizontal["TVT_input"].notna() & horizontal["GR"].notna()
    if int(known.sum()) < 20:
        return np.nan
    sample_pos = deterministic_sample_positions(known, 500)
    sample = horizontal.iloc[sample_pos]
    tw_gr = interp(sample["TVT_input"].to_numpy(float))
    if np.nanstd(tw_gr) < 1e-9 or np.nanstd(sample["GR"].to_numpy(float)) < 1e-9:
        return np.nan
    return float(np.corrcoef(sample["GR"].to_numpy(float), tw_gr)[0, 1])


def rmse_value(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def safe_corr(frame, feature, target, method):
    pair = frame[[feature, target]].replace([np.inf, -np.inf], np.nan).dropna()
    if len(pair) < 100 or pair[feature].nunique(dropna=True) < 2:
        return np.nan
    return float(pair[feature].corr(pair[target], method=method))


In [3]:
"Build sampled target-zone candidate features from train wells and visible local test wells."


def build_well_feature_sample(well, directory, include_target):
    horizontal = load_horizontal_well(directory / f"{well}__horizontal_well.csv")
    typewell_path = directory / f"{well}__typewell.csv"
    typewell = load_typewell(typewell_path) if typewell_path.exists() else pd.DataFrame()

    tvt_input = safe_numeric(horizontal["TVT_input"])
    md = safe_numeric(horizontal["MD"])
    gr_raw = safe_numeric(horizontal["GR"])
    gr_imputed = clean_gr(horizontal["GR"])
    gr_missing = gr_raw.isna().astype(float)
    target_mask = tvt_input.isna()
    if include_target:
        target_mask = target_mask & horizontal["TVT"].notna()
    known_positions = np.flatnonzero(tvt_input.notna().to_numpy())
    if len(known_positions) == 0 or int(target_mask.sum()) == 0:
        return pd.DataFrame()

    last_known_pos = int(known_positions[-1])
    sample_positions = deterministic_sample_positions(target_mask, MAX_TARGET_ROWS_PER_WELL)
    last_known_md = float(md.iloc[last_known_pos])
    last_known_tvt = float(tvt_input.iloc[last_known_pos])
    last_known_gr = float(gr_imputed.iloc[last_known_pos])
    last_known_z = float(safe_numeric(horizontal["Z"]).iloc[last_known_pos])
    typewell_corr = known_typewell_corr(horizontal, typewell)
    tw_interp = typewell_interpolator(typewell)

    slope_windows = [10, 25, 50, 100]
    tvt_slopes = {f"tvt_slope_last_{w}": window_slope(md, tvt_input, last_known_pos, w) for w in slope_windows}
    z_series = safe_numeric(horizontal["Z"])
    z_slopes = {f"z_slope_last_{w}": window_slope(md, z_series, last_known_pos, w) for w in slope_windows}

    gr_roll_mean_25 = gr_imputed.rolling(25, center=True, min_periods=1).mean()
    gr_roll_std_25 = gr_imputed.rolling(25, center=True, min_periods=2).std()
    gr_missing_frac_25 = gr_missing.rolling(25, center=True, min_periods=1).mean()
    gr_missing_frac_100 = gr_missing.rolling(100, center=True, min_periods=1).mean()
    dx = safe_numeric(horizontal["X"]) - float(safe_numeric(horizontal["X"]).iloc[last_known_pos])
    dy = safe_numeric(horizontal["Y"]) - float(safe_numeric(horizontal["Y"]).iloc[last_known_pos])
    dz = safe_numeric(horizontal["Z"]) - last_known_z
    horizontal_step = np.sqrt(safe_numeric(horizontal["X"]).diff().fillna(0.0) ** 2 + safe_numeric(horizontal["Y"]).diff().fillna(0.0) ** 2)
    horizontal_dist = horizontal_step.cumsum()
    horizontal_dist_from_boundary = horizontal_dist - float(horizontal_dist.iloc[last_known_pos])

    rows = []
    eval_positions = np.flatnonzero(target_mask.to_numpy())
    eval_rank = {int(pos): rank for rank, pos in enumerate(eval_positions)}
    eval_denominator = max(len(eval_positions) - 1, 1)
    for pos in sample_positions:
        distance_md = float(md.iloc[pos] - last_known_md)
        slope_50 = tvt_slopes["tvt_slope_last_50"]
        tvt_proxy = last_known_tvt + (0.0 if not np.isfinite(slope_50) else slope_50 * distance_md)
        tw_gr_proxy = float(tw_interp([tvt_proxy])[0]) if tw_interp is not None and np.isfinite(tvt_proxy) else np.nan
        tw_gr_last = float(tw_interp([last_known_tvt])[0]) if tw_interp is not None and np.isfinite(last_known_tvt) else np.nan
        row = {
            "wellname": well,
            "row_idx": int(horizontal.index[pos]),
            "MD": float(md.iloc[pos]),
            "X": float(safe_numeric(horizontal["X"]).iloc[pos]),
            "Y": float(safe_numeric(horizontal["Y"]).iloc[pos]),
            "Z": float(safe_numeric(horizontal["Z"]).iloc[pos]),
            "md_norm": float((md.iloc[pos] - md.min()) / (md.max() - md.min() + 1e-9)),
            "distance_from_boundary_md": distance_md,
            "eval_progress": float(eval_rank[int(pos)] / eval_denominator),
            "last_known_tvt": last_known_tvt,
            "last_known_md": last_known_md,
            "last_known_gr": last_known_gr,
            "last_known_z": last_known_z,
            "GR_imputed": float(gr_imputed.iloc[pos]),
            "GR_missing": float(gr_missing.iloc[pos]),
            "gr_roll_mean_25": float(gr_roll_mean_25.iloc[pos]),
            "gr_roll_std_25": float(gr_roll_std_25.iloc[pos]) if np.isfinite(gr_roll_std_25.iloc[pos]) else np.nan,
            "gr_missing_frac_25": float(gr_missing_frac_25.iloc[pos]),
            "gr_missing_frac_100": float(gr_missing_frac_100.iloc[pos]),
            "dX_from_boundary": float(dx.iloc[pos]),
            "dY_from_boundary": float(dy.iloc[pos]),
            "dZ_from_boundary": float(dz.iloc[pos]),
            "horizontal_dist_from_boundary": float(horizontal_dist_from_boundary.iloc[pos]),
            "tw_gr_at_proxy": tw_gr_proxy,
            "tw_gr_at_last_known": tw_gr_last,
            "gr_minus_tw_proxy": float(gr_imputed.iloc[pos] - tw_gr_proxy) if np.isfinite(tw_gr_proxy) else np.nan,
            "typewell_gr_corr": typewell_corr,
        }
        for key, value in tvt_slopes.items():
            row[key] = value
            row[f"{key}_projected_delta"] = value * distance_md if np.isfinite(value) else np.nan
        for key, value in z_slopes.items():
            row[key] = value
        if include_target:
            row["TVT"] = float(horizontal["TVT"].iloc[pos])
            row["delta_tvt_from_boundary"] = row["TVT"] - last_known_tvt
            row["endpoint_hold_error"] = row["TVT"] - last_known_tvt
            row["slope50_projection_error"] = row["TVT"] - (last_known_tvt + (0.0 if not np.isfinite(slope_50) else slope_50 * distance_md))
        rows.append(row)
    return pd.DataFrame(rows)

start = time.time()
train_wells = discover_wells(TRAIN_DIR)
test_wells = discover_wells(TEST_DIR)
train_feature_sample = pd.concat(
    [build_well_feature_sample(well, TRAIN_DIR, include_target=True) for well in train_wells],
    ignore_index=True,
)
test_feature_sample = pd.concat(
    [build_well_feature_sample(well, TEST_DIR, include_target=False) for well in test_wells],
    ignore_index=True,
)
build_seconds = time.time() - start

print({
    "train_wells": len(train_wells),
    "test_wells": len(test_wells),
    "train_sample_rows": len(train_feature_sample),
    "test_sample_rows": len(test_feature_sample),
    "build_seconds": round(build_seconds, 2),
})
train_feature_sample.head()


18:41:32 | INFO     | src.io.loaders | Discovered 773 wells in C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\data\raw\train


18:41:32 | INFO     | src.io.loaders | Discovered 3 wells in C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\data\raw\test


{'train_wells': 773, 'test_wells': 3, 'train_sample_rows': 193250, 'test_sample_rows': 750, 'build_seconds': 267.74}


,wellname,row_idx,MD,X,Y,Z,md_norm,distance_from_boundary_md,eval_progress,last_known_tvt,...,tvt_slope_last_100,tvt_slope_last_100_projected_delta,z_slope_last_10,z_slope_last_25,z_slope_last_50,z_slope_last_100,TVT,delta_tvt_from_boundary,endpoint_hold_error,slope50_projection_error
0,000d7d20,1442,12909.0,2983537.06,1070212.72,-9735.07,0.273261,1.0,0.000000,11747.37,...,0.008485,0.008485,0.007778,0.005833,0.002653,0.004343,11747.38,0.01,0.01,-0.006327
1,000d7d20,1457,12924.0,2983536.61,1070227.71,-9734.89,0.276104,16.0,0.003911,11747.37,...,0.008485,0.135758,0.007778,0.005833,0.002653,0.004343,11747.48,0.11,0.11,-0.151224
2,000d7d20,1472,12939.0,2983536.17,1070242.70,-9734.63,0.278946,31.0,0.007823,11747.37,...,0.008485,0.263030,0.007778,0.005833,0.002653,0.004343,11747.52,0.15,0.15,-0.356122
3,000d7d20,1488,12955.0,2983535.72,1070258.69,-9734.29,0.281978,47.0,0.011995,11747.37,...,0.008485,0.398788,0.007778,0.005833,0.002653,0.004343,11747.48,0.11,0.11,-0.657347
4,000d7d20,1503,12970.0,2983535.29,1070273.68,-9733.89,0.284821,62.0,0.015906,11747.37,...,0.008485,0.526061,0.007778,0.005833,0.002653,0.004343,11747.54,0.17,0.17,-0.842245


In [4]:
"Screen numeric candidate features against target-zone TVT and boundary-relative TVT delta."

excluded = {"TVT", "delta_tvt_from_boundary", "endpoint_hold_error", "slope50_projection_error"}
feature_columns = [
    col for col in train_feature_sample.columns
    if col not in excluded and col != "wellname" and pd.api.types.is_numeric_dtype(train_feature_sample[col])
]

signal_rows = []
for feature in feature_columns:
    signal_rows.append({
        "feature": feature,
        "non_null_frac": float(train_feature_sample[feature].notna().mean()),
        "n_unique": int(train_feature_sample[feature].nunique(dropna=True)),
        "pearson_tvt": safe_corr(train_feature_sample, feature, "TVT", "pearson"),
        "spearman_tvt": safe_corr(train_feature_sample, feature, "TVT", "spearman"),
        "pearson_delta": safe_corr(train_feature_sample, feature, "delta_tvt_from_boundary", "pearson"),
        "spearman_delta": safe_corr(train_feature_sample, feature, "delta_tvt_from_boundary", "spearman"),
    })
feature_signal = pd.DataFrame(signal_rows)

mi_frame = train_feature_sample[feature_columns + ["delta_tvt_from_boundary"]].replace([np.inf, -np.inf], np.nan).dropna(subset=["delta_tvt_from_boundary"])
if len(mi_frame) > MAX_MI_ROWS:
    mi_frame = mi_frame.sample(MAX_MI_ROWS, random_state=SEED)
mi_X = mi_frame[feature_columns].replace([np.inf, -np.inf], np.nan)
mi_X = pd.DataFrame(SimpleImputer(strategy="median").fit_transform(mi_X), columns=feature_columns)
mi_y = mi_frame["delta_tvt_from_boundary"].to_numpy(float)
mi_scores = mutual_info_regression(mi_X, mi_y, random_state=SEED, n_neighbors=5)
mi_table = pd.DataFrame({"feature": feature_columns, "mutual_info_delta": mi_scores})
feature_signal = feature_signal.merge(mi_table, on="feature", how="left")
feature_signal["abs_spearman_delta"] = feature_signal["spearman_delta"].abs()
feature_signal = feature_signal.sort_values(["abs_spearman_delta", "mutual_info_delta"], ascending=False)

signal_path = TABLE_DIR / f"eda__phase3_candidate_feature_signal__{RUN_DATE}.csv"
feature_signal.to_csv(signal_path, index=False)
feature_signal.head(15)


,feature,non_null_frac,n_unique,pearson_tvt,spearman_tvt,pearson_delta,spearman_delta,mutual_info_delta,abs_spearman_delta
26,tvt_slope_last_10,1.0,216,0.012000,0.007843,0.243064,0.198728,0.331993,0.198728
28,tvt_slope_last_25,1.0,381,0.005847,0.001363,0.228097,0.182750,0.463316,0.182750
34,z_slope_last_10,1.0,239,-0.034022,-0.038539,-0.223333,-0.166966,0.397059,0.166966
27,tvt_slope_last_10_projected_delta,1.0,152374,0.003592,0.009757,0.204369,0.163694,0.283463,0.163694
30,tvt_slope_last_50,1.0,498,0.004194,-0.004335,0.221266,0.161217,0.485306,0.161217
35,z_slope_last_25,1.0,434,-0.032238,-0.038056,-0.218897,-0.160726,0.497100,0.160726
36,z_slope_last_50,1.0,562,-0.028695,-0.035218,-0.213061,-0.150834,0.506849,0.150834
32,tvt_slope_last_100,1.0,601,-0.011208,-0.024780,0.217051,0.149294,0.493143,0.149294
29,tvt_slope_last_25_projected_delta,1.0,175442,-0.000641,0.006168,0.189905,0.147510,0.276114,0.147510
37,z_slope_last_100,1.0,654,-0.020486,-0.028285,-0.211150,-0.142901,0.513772,0.142901


In [5]:
"Run grouped diagnostic linear probes on feature families."

feature_groups = {
    "endpoint_hold": [],
    "slope50_projection": [],
    "trajectory": ["MD", "X", "Y", "Z", "md_norm", "dX_from_boundary", "dY_from_boundary", "dZ_from_boundary", "horizontal_dist_from_boundary"],
    "boundary_endpoint": ["row_idx", "MD", "md_norm", "distance_from_boundary_md", "eval_progress", "last_known_tvt", "last_known_md", "last_known_z"],
    "recent_slopes": ["distance_from_boundary_md", "eval_progress", "last_known_tvt", "tvt_slope_last_10", "tvt_slope_last_25", "tvt_slope_last_50", "tvt_slope_last_100", "tvt_slope_last_10_projected_delta", "tvt_slope_last_25_projected_delta", "tvt_slope_last_50_projected_delta", "tvt_slope_last_100_projected_delta"],
    "gr_availability": ["last_known_tvt", "distance_from_boundary_md", "GR_imputed", "GR_missing", "last_known_gr", "gr_roll_mean_25", "gr_roll_std_25", "gr_missing_frac_25", "gr_missing_frac_100"],
    "typewell_alignment": ["last_known_tvt", "distance_from_boundary_md", "tw_gr_at_proxy", "tw_gr_at_last_known", "gr_minus_tw_proxy", "typewell_gr_corr"],
}
all_candidate = sorted(set(sum((cols for cols in feature_groups.values()), [])))
feature_groups["all_candidate"] = all_candidate

probe_rows = []
folds = group_well_folds(sorted(train_feature_sample["wellname"].unique()), n_splits=N_SPLITS)
for fold_idx, (train_fold_wells, valid_fold_wells) in enumerate(folds, start=1):
    train_mask = train_feature_sample["wellname"].isin(train_fold_wells)
    valid_mask = train_feature_sample["wellname"].isin(valid_fold_wells)
    valid_frame = train_feature_sample.loc[valid_mask].copy()
    for group_name, columns in feature_groups.items():
        if group_name == "endpoint_hold":
            pred_tvt = valid_frame["last_known_tvt"].to_numpy(float)
        elif group_name == "slope50_projection":
            slope = valid_frame["tvt_slope_last_50"].fillna(0.0).to_numpy(float)
            pred_tvt = valid_frame["last_known_tvt"].to_numpy(float) + slope * valid_frame["distance_from_boundary_md"].to_numpy(float)
        else:
            train_frame = train_feature_sample.loc[train_mask].copy()
            model = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), Ridge(alpha=10.0))
            model.fit(train_frame[columns], train_frame["delta_tvt_from_boundary"])
            pred_delta = model.predict(valid_frame[columns])
            pred_tvt = valid_frame["last_known_tvt"].to_numpy(float) + pred_delta
        y_true = valid_frame["TVT"].to_numpy(float)
        probe_rows.append({
            "fold": fold_idx,
            "feature_group": group_name,
            "n_valid_rows": int(len(valid_frame)),
            "rmse": rmse_value(y_true, pred_tvt),
            "mae": float(mean_absolute_error(y_true, pred_tvt)),
            "r2": float(r2_score(y_true, pred_tvt)),
        })

probe_metrics = pd.DataFrame(probe_rows)
probe_summary = (
    probe_metrics.groupby("feature_group", as_index=False)
    .agg(rmse_mean=("rmse", "mean"), rmse_std=("rmse", "std"), mae_mean=("mae", "mean"), r2_mean=("r2", "mean"), n_valid_rows=("n_valid_rows", "sum"))
    .sort_values("rmse_mean")
)

probe_metrics_path = TABLE_DIR / f"eda__phase3_feature_group_probe_metrics__{RUN_DATE}.csv"
probe_summary_path = TABLE_DIR / f"eda__phase3_feature_group_probe_summary__{RUN_DATE}.csv"
probe_metrics.to_csv(probe_metrics_path, index=False)
probe_summary.to_csv(probe_summary_path, index=False)
probe_summary


,feature_group,rmse_mean,rmse_std,mae_mean,r2_mean,n_valid_rows
0,all_candidate,14.623597,0.728165,10.329438,0.999443,193250
6,trajectory,14.868880,0.630836,10.535242,0.999424,193250
4,recent_slopes,15.113580,0.548559,10.593526,0.999404,193250
3,gr_availability,15.532202,0.577748,10.920176,0.999371,193250
7,typewell_alignment,15.535416,0.570312,10.932725,0.999370,193250
1,boundary_endpoint,15.540509,0.552881,10.917754,0.999370,193250
2,endpoint_hold,15.581522,0.653451,10.916724,0.999367,193250
5,slope50_projection,104.532172,14.839352,51.108900,0.971208,193250


In [6]:
"Compare train target-zone candidates with visible local test rows as a diagnostic caveat only."

drift_features = [
    "MD", "md_norm", "distance_from_boundary_md", "eval_progress", "GR_imputed", "GR_missing",
    "gr_missing_frac_25", "gr_missing_frac_100", "tw_gr_at_proxy", "gr_minus_tw_proxy", "typewell_gr_corr",
]

drift_rows = []
for feature in drift_features:
    if feature not in test_feature_sample.columns:
        continue
    train_values = train_feature_sample[feature].replace([np.inf, -np.inf], np.nan).dropna()
    test_values = test_feature_sample[feature].replace([np.inf, -np.inf], np.nan).dropna()
    if len(train_values) < 10 or len(test_values) < 10:
        continue
    ks = ks_2samp(train_values, test_values)
    drift_rows.append({
        "feature": feature,
        "train_mean": float(train_values.mean()),
        "test_mean": float(test_values.mean()),
        "train_median": float(train_values.median()),
        "test_median": float(test_values.median()),
        "ks_statistic": float(ks.statistic),
        "ks_pvalue": float(ks.pvalue),
    })

drift_table = pd.DataFrame(drift_rows).sort_values("ks_statistic", ascending=False)
drift_path = TABLE_DIR / f"eda__phase3_visible_test_drift__{RUN_DATE}.csv"
drift_table.to_csv(drift_path, index=False)

gr_missing_summary = (
    train_feature_sample.assign(GR_missing_bucket=train_feature_sample["GR_missing"].map({0.0: "GR present", 1.0: "GR missing"}))
    .groupby("GR_missing_bucket", as_index=False)
    .agg(rows=("TVT", "size"), delta_mean=("delta_tvt_from_boundary", "mean"), delta_median=("delta_tvt_from_boundary", "median"), tvt_mean=("TVT", "mean"), tvt_std=("TVT", "std"))
)
gr_missing_path = TABLE_DIR / f"eda__phase3_gr_missing_signal__{RUN_DATE}.csv"
gr_missing_summary.to_csv(gr_missing_path, index=False)

drift_table.head(10)


,feature,train_mean,test_mean,train_median,test_median,ks_statistic,ks_pvalue
10,typewell_gr_corr,0.801877,0.846469,0.812477,0.823135,0.350582,1.378948e-82
4,GR_imputed,88.093538,73.912444,86.339726,83.185268,0.276721,4.657973e-51
7,gr_missing_frac_100,0.314848,0.236588,0.300000,0.150000,0.242441,3.889678e-39
6,gr_missing_frac_25,0.314756,0.235963,0.280000,0.160000,0.211208,1.023320e-29
9,gr_minus_tw_proxy,7.515653,-10.027537,4.864981,-6.627252,0.210910,1.240175e-29
8,tw_gr_at_proxy,80.577885,83.939981,80.935233,85.134257,0.145892,2.434599e-14
0,MD,15039.529144,15214.504000,14888.000000,15107.000000,0.075167,4.071424e-04
5,GR_missing,0.313113,0.241333,0.000000,0.000000,0.071779,8.598200e-04
2,distance_from_boundary_md,2447.608057,2358.504000,2283.000000,2275.000000,0.048026,6.169535e-02
1,md_norm,0.633364,0.633924,0.636293,0.635385,0.012521,9.997280e-01


In [7]:
"Create Phase 3 diagnostic figures."

fig_paths = {}

top_signal = feature_signal.head(15).sort_values("abs_spearman_delta")
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top_signal["feature"], top_signal["abs_spearman_delta"], color="#4C78A8")
ax.set_title("Phase 3 Feature Signal Rank")
ax.set_xlabel("absolute Spearman correlation with target-zone TVT delta")
ax.set_ylabel("")
fig.tight_layout()
fig_paths["feature_signal_rank"] = FIGURE_DIR / f"eda__phase3_feature_signal_rank__{RUN_DATE}.png"
fig.savefig(fig_paths["feature_signal_rank"], dpi=160)
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 5))
ordered_probe = probe_summary.sort_values("rmse_mean", ascending=True)
ax.barh(ordered_probe["feature_group"], ordered_probe["rmse_mean"], color="#59A14F")
ax.set_title("Diagnostic Linear Probe RMSE by Feature Family")
ax.set_xlabel("grouped target-zone RMSE")
ax.set_ylabel("")
fig.tight_layout()
fig_paths["group_probe_rmse"] = FIGURE_DIR / f"eda__phase3_group_probe_rmse__{RUN_DATE}.png"
fig.savefig(fig_paths["group_probe_rmse"], dpi=160)
plt.close(fig)

boundary_bins = train_feature_sample.copy()
boundary_bins["distance_bin"] = pd.qcut(boundary_bins["distance_from_boundary_md"], q=20, duplicates="drop")
boundary_curve = boundary_bins.groupby("distance_bin", observed=True).agg(distance_mid=("distance_from_boundary_md", "mean"), delta_mean=("delta_tvt_from_boundary", "mean"), delta_p25=("delta_tvt_from_boundary", lambda s: np.percentile(s, 25)), delta_p75=("delta_tvt_from_boundary", lambda s: np.percentile(s, 75))).reset_index()
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(boundary_curve["distance_mid"], boundary_curve["delta_mean"], marker="o", color="#E15759")
ax.fill_between(boundary_curve["distance_mid"], boundary_curve["delta_p25"], boundary_curve["delta_p75"], color="#E15759", alpha=0.18)
ax.set_title("Target-Zone TVT Delta Grows with Boundary Distance")
ax.set_xlabel("MD distance past last known TVT_input")
ax.set_ylabel("TVT - last known TVT_input")
fig.tight_layout()
fig_paths["boundary_delta_curve"] = FIGURE_DIR / f"eda__phase3_boundary_delta_curve__{RUN_DATE}.png"
fig.savefig(fig_paths["boundary_delta_curve"], dpi=160)
plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(gr_missing_summary["GR_missing_bucket"], gr_missing_summary["delta_median"], color=["#F28E2B", "#B07AA1"][: len(gr_missing_summary)])
ax.set_title("Target-Zone Delta by GR Availability")
ax.set_xlabel("")
ax.set_ylabel("median TVT delta from boundary")
fig.tight_layout()
fig_paths["gr_missing_signal"] = FIGURE_DIR / f"eda__phase3_gr_missing_signal__{RUN_DATE}.png"
fig.savefig(fig_paths["gr_missing_signal"], dpi=160)
plt.close(fig)

typewell_plot = train_feature_sample[["gr_minus_tw_proxy", "delta_tvt_from_boundary"]].replace([np.inf, -np.inf], np.nan).dropna()
if len(typewell_plot) > 60000:
    typewell_plot = typewell_plot.sample(60000, random_state=SEED)
fig, ax = plt.subplots(figsize=(7, 5))
ax.hexbin(typewell_plot["gr_minus_tw_proxy"], typewell_plot["delta_tvt_from_boundary"], gridsize=45, cmap="viridis", mincnt=1)
ax.set_title("Typewell GR Residual vs Target-Zone Delta")
ax.set_xlabel("horizontal GR - typewell GR at slope-proxy TVT")
ax.set_ylabel("TVT - last known TVT_input")
fig.tight_layout()
fig_paths["typewell_residual_signal"] = FIGURE_DIR / f"eda__phase3_typewell_residual_signal__{RUN_DATE}.png"
fig.savefig(fig_paths["typewell_residual_signal"], dpi=160)
plt.close(fig)

{key: str(value) for key, value in fig_paths.items()}


{'feature_signal_rank': 'C:\\Users\\sardo\\OneDrive\\Desktop\\Classes\\Projects\\Wellbore-tvt-Modeling\\artifacts\\figures\\eda__phase3_feature_signal_rank__2026-06-21.png',
 'group_probe_rmse': 'C:\\Users\\sardo\\OneDrive\\Desktop\\Classes\\Projects\\Wellbore-tvt-Modeling\\artifacts\\figures\\eda__phase3_group_probe_rmse__2026-06-21.png',
 'boundary_delta_curve': 'C:\\Users\\sardo\\OneDrive\\Desktop\\Classes\\Projects\\Wellbore-tvt-Modeling\\artifacts\\figures\\eda__phase3_boundary_delta_curve__2026-06-21.png',
 'gr_missing_signal': 'C:\\Users\\sardo\\OneDrive\\Desktop\\Classes\\Projects\\Wellbore-tvt-Modeling\\artifacts\\figures\\eda__phase3_gr_missing_signal__2026-06-21.png',
 'typewell_residual_signal': 'C:\\Users\\sardo\\OneDrive\\Desktop\\Classes\\Projects\\Wellbore-tvt-Modeling\\artifacts\\figures\\eda__phase3_typewell_residual_signal__2026-06-21.png'}

In [8]:
"Write findings, report, and run bundle."

best_probe = probe_summary.iloc[0]
endpoint_probe = probe_summary.loc[probe_summary["feature_group"] == "endpoint_hold"].iloc[0]
slope_probe = probe_summary.loc[probe_summary["feature_group"] == "slope50_projection"].iloc[0]
top_corr = feature_signal.iloc[0]
top_mi = feature_signal.sort_values("mutual_info_delta", ascending=False).iloc[0]
grt = gr_missing_summary.set_index("GR_missing_bucket")
if {"GR present", "GR missing"}.issubset(grt.index):
    gr_delta_gap = float(grt.loc["GR missing", "delta_median"] - grt.loc["GR present", "delta_median"])
else:
    gr_delta_gap = np.nan
max_drift = drift_table.iloc[0] if len(drift_table) else pd.Series({"feature": "none", "ks_statistic": np.nan})

findings = pd.DataFrame([
    {
        "topic": "boundary_endpoint_signal",
        "finding": f"Endpoint hold diagnostic RMSE is {endpoint_probe.rmse_mean:.4f}; the best feature-family probe is {best_probe.feature_group} with RMSE {best_probe.rmse_mean:.4f}.",
        "modeling_implication": "Boundary and endpoint features should be first-class inputs for boosted trees and Bayesian generative models.",
    },
    {
        "topic": "recent_slope_projection",
        "finding": f"Last-50 slope projection diagnostic RMSE is {slope_probe.rmse_mean:.4f}.",
        "modeling_implication": "Recent slopes are useful context, but should be combined with endpoint, position, and other well-relative features rather than used as a standalone extrapolator.",
    },
    {
        "topic": "feature_screening",
        "finding": f"Top absolute Spearman feature for target-zone delta is {top_corr.feature} ({top_corr.abs_spearman_delta:.3f}); top mutual-information feature is {top_mi.feature} ({top_mi.mutual_info_delta:.3f}).",
        "modeling_implication": "Use the ranked feature table to choose candidate groups for the richer feature builder instead of guessing from raw columns.",
    },
    {
        "topic": "gr_availability_signal",
        "finding": f"Median target-zone delta gap between GR-missing and GR-present sampled rows is {gr_delta_gap:.4f}.",
        "modeling_implication": "GR missingness indicators and rolling availability summaries should remain in tree/boosted feature candidates.",
    },
    {
        "topic": "typewell_signal",
        "finding": f"Typewell residual feature gr_minus_tw_proxy has Spearman delta correlation {safe_corr(train_feature_sample, 'gr_minus_tw_proxy', 'delta_tvt_from_boundary', 'spearman'):.3f}.",
        "modeling_implication": "Typewell residuals should be tested as robust, smoothed features rather than raw matching signals.",
    },
    {
        "topic": "visible_test_drift_caveat",
        "finding": f"Largest visible train/test descriptor KS statistic is {max_drift.ks_statistic:.3f} for {max_drift.feature}, but the visible test wells overlap train wells.",
        "modeling_implication": "Use visible-test drift diagnostics only for notebook robustness and submission-shape checks; Phase 4 should evaluate held-out fold similarity instead.",
    },
])

export_sample = pd.concat([
    group.iloc[np.linspace(0, len(group) - 1, min(len(group), MAX_EXPORT_ROWS_PER_WELL)).astype(int)]
    for _, group in train_feature_sample.groupby("wellname", sort=True)
], ignore_index=True)
sample_path = TABLE_DIR / f"eda__phase3_candidate_feature_sample_preview__{RUN_DATE}.csv"
findings_path = TABLE_DIR / f"eda__phase3_findings__{RUN_DATE}.csv"
export_sample.to_csv(sample_path, index=False)
findings.to_csv(findings_path, index=False)

report_path = REPORT_DIR / f"eda__phase3_feature_signal__{RUN_DATE}.md"
report_lines = [
    "# Phase 3 Feature Signal EDA",
    "",
    f"Date: {RUN_DATE}",
    "",
    "## Key Findings",
]
for row in findings.itertuples(index=False):
    report_lines.extend([
        f"- **{row.topic}**: {row.finding}",
        f"  - Modeling implication: {row.modeling_implication}",
    ])
report_lines.extend([
    "",
    "## Artifacts",
    f"- Candidate feature sample: {sample_path}",
    f"- Candidate feature signal: {signal_path}",
    f"- Feature group probe metrics: {probe_metrics_path}",
    f"- Feature group probe summary: {probe_summary_path}",
    f"- Visible test drift: {drift_path}",
    f"- GR missingness signal: {gr_missing_path}",
    f"- Findings table: {findings_path}",
])
for name, fig_path in fig_paths.items():
    report_lines.append(f"- {name}: {fig_path}")
report_path.write_text("\n".join(report_lines) + "\n", encoding="utf-8")

config = {
    "task": "phase3_feature_signal_eda",
    "seed": SEED,
    "max_target_rows_per_well": MAX_TARGET_ROWS_PER_WELL,
    "max_export_rows_per_well": MAX_EXPORT_ROWS_PER_WELL,
    "max_mi_rows": MAX_MI_ROWS,
    "n_splits": N_SPLITS,
    "note": "Diagnostic target-zone feature screening; not a submission model.",
}
with RunBundle(RUN_TAG, config, seed=SEED) as rb:
    rb.add_input(str(TRAIN_DIR))
    rb.add_input(str(TEST_DIR))
    rb.add_metric("train_wells", float(len(train_wells)))
    rb.add_metric("test_wells", float(len(test_wells)))
    rb.add_metric("train_sample_rows", float(len(train_feature_sample)))
    rb.add_metric("test_sample_rows", float(len(test_feature_sample)))
    rb.add_metric("export_sample_rows", float(len(export_sample)))
    rb.add_metric("best_probe_rmse", float(best_probe.rmse_mean))
    rb.add_metric("endpoint_hold_rmse", float(endpoint_probe.rmse_mean))
    rb.add_metric("slope50_projection_rmse", float(slope_probe.rmse_mean))
    rb.add_metric("top_abs_spearman_delta", float(top_corr.abs_spearman_delta))
    rb.add_output(str(sample_path))
    rb.add_output(str(signal_path))
    rb.add_output(str(probe_metrics_path))
    rb.add_output(str(probe_summary_path))
    rb.add_output(str(drift_path))
    rb.add_output(str(gr_missing_path))
    rb.add_output(str(findings_path))
    rb.add_output(str(report_path))
    for fig_path in fig_paths.values():
        rb.add_output(str(fig_path))
    rb.set_notes("Phase 3 feature-signal EDA completed. Linear probes are diagnostic and use sampled target-zone rows from grouped held-out wells.")
run_id = rb.run_id

print({
    "run_id": run_id,
    "best_probe": best_probe.feature_group,
    "best_probe_rmse": round(float(best_probe.rmse_mean), 4),
    "endpoint_hold_rmse": round(float(endpoint_probe.rmse_mean), 4),
    "top_spearman_feature": top_corr.feature,
    "top_spearman_abs": round(float(top_corr.abs_spearman_delta), 4),
})
findings


{'run_id': '2026-06-21__1847__eda_phase3_feature_signal__ebaa6c3', 'best_probe': 'all_candidate', 'best_probe_rmse': 14.6236, 'endpoint_hold_rmse': 15.5815, 'top_spearman_feature': 'tvt_slope_last_10', 'top_spearman_abs': 0.1987}


,topic,finding,modeling_implication
0,boundary_endpoint_signal,Endpoint hold diagnostic RMSE is 15.5815; the ...,Boundary and endpoint features should be first...
1,recent_slope_projection,Last-50 slope projection diagnostic RMSE is 10...,"Recent slopes are useful context, but should b..."
2,feature_screening,Top absolute Spearman feature for target-zone ...,Use the ranked feature table to choose candida...
3,gr_availability_signal,Median target-zone delta gap between GR-missin...,GR missingness indicators and rolling availabi...
4,typewell_signal,Typewell residual feature gr_minus_tw_proxy ha...,"Typewell residuals should be tested as robust,..."
5,visible_test_drift_caveat,Largest visible train/test descriptor KS stati...,Use visible-test drift diagnostics only for no...


## Results

The executed notebook writes the Phase 3 tables, figures, markdown report, and run bundle under `artifacts/`. The findings table at the end is the compact source of truth for context updates.

## Takeaways

Use the candidate feature signal and diagnostic probe tables to decide which richer feature families should be implemented before boosted-tree tuning. Treat visible local test comparisons as caveats only because those wells overlap train.
